<font color='darkred'> Unless otherwise noted, **this notebook will not be reviewed or autograded.**</font> You are welcome to use it for scratchwork, but **only the files listed in the exercises will be checked.**

---

# Exercises

For these exercises, add your functions to the *apputil\.py* file. If you like, you're welcome to adjust the *app\.py* file, but it is not required.

## Notes on Recursion

A [recursive function](https://www.w3schools.com/python/gloss_python_function_recursion.asp) is one which calls itself.

1. When the function is called, your CPU runs through each line of code until the function needs to be called again.
2. At that point, all variables are saved in memory, and the function runs through each line of code again until the function is called (again, but with a different passed argument), and so on.
3. Eventually, this process will stop at the "bottom of the **stack**", where the function doesn't get a chance to call itself again (likely because of some condition un/met by the latest passed argument).
4. Then, your CPU will work its way back up the stack to the final result. For example, take a look at [this visual example](https://realpython.com/python-recursion/#calculate-factorial) of calculating 4!.

When you write these functions, keep two things in mind:

- You will need a built-in stopping point (i.e., the "bottom"), where your function returns some result before it calls itself.
- **Don't think too hard about this.** Recursion can be perplexing to conceptualize when writing the code. So, when you call the function inside the function, think about it as a magical "hidden" function that has already done what you want it to do.
- [Python Tutor](https://pythontutor.com/) ([editor](https://pythontutor.com/visualize.html#mode=edit)) can be a helpful resource for this exercise!

## Exercise 1

The Fibonacci Series is credited to Leonardo Bonacci (_**fi**lius Bonacci_, 'son of Bonacci'), but can be traced back to early Indian mathematicians (circa 400BC). Starting with $\{0, \ 1\}$, each of the following numbers are the sum of the previous two numbers in the series.

`0 1 1 2 3 5 8 13 21 34 ...`

So, `fibonacci(9) = 34`. *Note: If the sequence starts with $\{2,\ 1\}$, it is then the [Lucas Sequence](https://en.wikipedia.org/wiki/Lucas_number).*

Write a recursive function (`fibonacci`) that, given `n`, will return the `n`th number of the Fibonacci Series.

*Test your function using Google or any other tool that can calculate the Fibonacci Series.*

In [1]:
import numpy as np
import pandas as pd

def fibonacci(n):
    """Return the nth number of the Fibonacci series."""
 
    # base case: fibonacci(0) is 0, fibonacci(1) is 1
    if n < 2:
        return n
 
    # no print in here. this calls itself thousands of times and
    # would flood the screen.
    return fibonacci(n - 1) + fibonacci(n - 2)


## Exercise 2

Write a (single) recursive function, `to_binary()`, that [converts](https://en.wikipedia.org/wiki/Binary_number#Conversion_to_and_from_other_numeral_systems) an integer into its [binary](https://en.wikipedia.org/wiki/Binary_number) representation. So, for example:

```python
to_binary(2)   -->  10
to_binary(12)  -->  1100
```

*Note: you can test your function with the built in `bin()` function.*

In [3]:
def to_binary(n):
    """Return the binary representation of an integer."""
 
    # 0 and 1 are already binary
    if n < 2:
        return n
 
    # n % 2 is the last binary digit and n // 2 is everything before
    # it. multiply the front part by 10 to make room for that digit.
    return to_binary(n // 2) * 10 + n % 2
 

## Exercise 3 

Use the raw Bellevue Almshouse Dataset (`df_bellevue`) extracted at the top of the lab (i.e., with `pd.read_csv ...`).

**Write a function for each of the following tasks. Name these functions `task_i()`** (i.e., without any input arguments).

1. Return a list of all column names, *sorted* such that the first column has the *least* missing values, and the last column has the *most* missing values (use the raw column names).
   - *Note: there is an issue with the `gender` column you'll need to remedy first ...*
2. Return a **data frame** with two columns:
   - the year (for each year in the data), `year`
   - the total number of entries (immigrant admissions) for each year, `total_admissions`
3. Return a **series** with:
   - Index: gender (for each gender in the data)
   - Values: the average age for the indexed gender.
4. Return a list of the 5 most common professions *in order of prevalence* (so, the most common is first).

For each of these, if there are messy data issues, use the `print` statement to explain.


In [4]:
URL = ("https://raw.githubusercontent.com/melaniewalsh/"
       "Intro-Cultural-Analytics/master/book/data/"
       "bellevue_almshouse_modified.csv")

df_bellevue = pd.read_csv(URL)

def clean_gender(df):
    """Replace the junk values in the gender column with NaN."""
 
    # gender should only be 'm' or 'w'. '?', 'g' and 'h' are typos,
    # and pandas won't count them as missing unless we swap them out.
    df = df.copy()
    df['gender'] = df['gender'].replace(['?', 'g', 'h'], np.nan)
    return df
 
 
def task_1():
    """List the column names, fewest missing values first."""
 
    print("--- Exercise 3, task 1 ---")
 
    print("Messy data: gender shows 0 missing values, but look at "
          "what's in it:")
    print(df_bellevue['gender'].value_counts().to_string())
    print("'?', 'g' and 'h' are typos. Pandas treats them as real "
          "values because they're strings, not NaN.")
    print("After cleaning, gender goes from 0 missing to 5.")
 
    df = clean_gender(df_bellevue)
    missing = df.isna().sum().sort_values()
 
    print("Missing values per column, after cleaning:")
    print(missing.to_string())
 
    return list(missing.index)
 
 
def task_2():
    """Count the admissions for each year in the data."""
 
    print("--- Exercise 3, task 2 ---")
 
    df = df_bellevue.copy()
 
    print(f"Messy data: date_in is stored as {df['date_in'].dtype}, "
          f"not a date, so it gets converted first.")
    df['year'] = pd.to_datetime(df['date_in']).dt.year
 
    counts = df.groupby('year').size()
    result = counts.reset_index(name='total_admissions')
 
    print(f"Found {len(result)} years in the data.")
 
    return result
 
 
def task_3():
    """Return the average age for each gender."""
 
    print("--- Exercise 3, task 3 ---")
 
    missing_age = df_bellevue['age'].isna().sum()
    print(f"Messy data: {missing_age} rows have no age. mean() "
          f"skips them.")
    print("Junk gender values are dropped first, so only 'm' and "
          "'w' are averaged.")
 
    df = clean_gender(df_bellevue)
    return df.groupby('gender')['age'].mean()
 
 
def task_4():
    """Return the 5 most common professions, most common first."""
 
    print("--- Exercise 3, task 4 ---")
 
    missing_job = df_bellevue['profession'].isna().sum()
    print(f"Messy data: {missing_job} rows have no profession.")
    print("'married', 'spinster' and 'widow' aren't jobs either. For "
          "a lot of the women, the clerk wrote down marital status "
          "instead of a profession.")
 
    # value_counts() already sorts most common first
    counts = df_bellevue['profession'].value_counts()
    return list(counts.head(5).index)
 

In [5]:
from IPython.display import display

# a list of the functions themselves, so we can loop over them
# instead of typing four near-identical lines.
tasks = [task_1, task_2, task_3, task_4]

for number, task in enumerate(tasks, start=1):
    print("=" * 60)
    print(f"TASK {number}: {task.__doc__}")
    print("=" * 60)

    result = task()

    print(f"\nReturned a {type(result).__name__}:")
    display(result)
    print()

TASK 1: List the column names, fewest missing values first.
--- Exercise 3, task 1 ---
Messy data: gender shows 0 missing values, but look at what's in it:
gender
m    4958
w    4621
?       2
g       2
h       1
'?', 'g' and 'h' are typos. Pandas treats them as real values because they're strings, not NaN.
After cleaning, gender goes from 0 missing to 5.
Missing values per column, after cleaning:
date_in          0
last_name        0
first_name       4
gender           5
age             50
profession    1019
disease       3087
children      9547

Returned a list:


['date_in',
 'last_name',
 'first_name',
 'gender',
 'age',
 'profession',
 'disease',
 'children']


TASK 2: Count the admissions for each year in the data.
--- Exercise 3, task 2 ---
Messy data: date_in is stored as str, not a date, so it gets converted first.
Found 2 years in the data.

Returned a DataFrame:


,year,total_admissions
0,1846,3073
1,1847,6511



TASK 3: Return the average age for each gender.
--- Exercise 3, task 3 ---
Messy data: 50 rows have no age. mean() skips them.
Junk gender values are dropped first, so only 'm' and 'w' are averaged.

Returned a Series:


gender
m    31.813433
w    28.725162
Name: age, dtype: float64


TASK 4: Return the 5 most common professions, most common first.
--- Exercise 3, task 4 ---
Messy data: 1019 rows have no profession.
'married', 'spinster' and 'widow' aren't jobs either. For a lot of the women, the clerk wrote down marital status instead of a profession.

Returned a list:


['laborer', 'married', 'spinster', 'widow', 'shoemaker']

## (Optional) Bonus Exercise 4

[Memoization](https://en.wikipedia.org/wiki/Memoization) is a technique where you store the results of expensive function calls and return the cached result when the same inputs occur again. This is a form of dynamic programming, and we can apply it to either of the above recursive functions to improve efficiency.

Write a memoized version of either the `fibonacci` or `to_binary` function above. *Hint: consider using a `global`ly defined `defaultdict`.*